# 02 — Feature embedding (`feature_embedding.py`)

This stage turns extracted numerical features into the representations carried by the network: `m` for the MSA, `z` for residue pairs, and `e` for the extra MSA. Everything here is projection and broadcasting; attention begins in the Evoformer.

## Stage map

```text
target_feat --Linear_i--+                         +--> pair z (R,R,c_z)
                        +-- outer sum + relpos ----+
target_feat --Linear_j--+

msa_feat ------- Linear + broadcast target -------> MSA m (S,R,c_m)
extra_msa_feat -------- Linear --------------------> extra MSA e (E,R,c_e)

previous m[0] --LayerNorm--+
previous z    --LayerNorm--+--> additions used by the next recycle
```

**Read alongside:** `../src/af2_from_scratch/feature_embedding.py`. Embedding changes feature channels and creates pairwise structure; it does not perform attention.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
from af2_from_scratch.feature_extraction import msa_features, sample_batch
from af2_from_scratch.feature_embedding import InputEmbedder
from af2_from_scratch import AF2Config

cfg = AF2Config()
f = msa_features("../examples/tautomerase/alignment.a3m")
b = sample_batch(f, cfg.n_clu, cfg.n_ext, seed=0)
emb = InputEmbedder(cfg)

## 1. The pair representation starts as an OUTER SUM
`z = tf_i(target)[:,None] + tf_j(target)[None]` — the row-copy broadcasting against the column-copy. Every cell z_ij mixes 'residue i as a row' with 'residue j as a column'. This is the each-with-each pattern from the tensor intro, producing the grid shape (R, R).

In [ ]:
t = b["target_feat"]
a, c = emb.tf_i(t), emb.tf_j(t)
z_outer = a[:, None] + c[None]
print(
    "tf_i(t):",
    tuple(a.shape),
    " +  tf_j(t):",
    tuple(c.shape),
    " ->  z:",
    tuple(z_outer.shape),
)
plt.imshow(z_outer[..., 0].detach(), cmap="RdBu", vmin=-1, vmax=1)
plt.title("z[:,:,0]: every cell = f(residue i) + g(residue j)")
plt.colorbar()
plt.show()

## 2. Relative position encoding (Algorithm 4)
`relpos` tells z how far apart residues are *in the sequence*: d_ij = clamp(i−j) one-hotted and projected. The diagonal structure is visible immediately.

In [ ]:
rp = emb.relpos(b["residue_index"])
plt.imshow(rp[..., 0].detach(), cmap="viridis")
plt.title("relpos channel 0: sequence distance i-j (diagonal!)")
plt.colorbar()
plt.show()

## 3. Full embedder
`m` = Linear(msa_feat) + query embedding broadcast to every row. `e` = cheaper embedding for the extra MSA (fewer channels, c_e=32).

In [ ]:
m, z, e = emb(b)
print("m:", tuple(m.shape), "  z:", tuple(z.shape), "  e:", tuple(e.shape))
print(f"embedder params: {sum(p.numel() for p in emb.parameters()) / 1e3:.0f}k")

## 4. Recycling (Algorithm 32)
AlphaFold runs the trunk multiple times, feeding LayerNormed outputs back as inputs. The first pass has no predecessor; later passes refine. `RecyclingEmbedder` is just two LayerNorms — gradients are detached across recycles.

In [ ]:
from af2_from_scratch.feature_embedding import RecyclingEmbedder

rec = RecyclingEmbedder(cfg)
m2, z2 = rec(m, z, m[0].detach(), z.detach())  # what recycle 2 would see
print("recycled shapes unchanged:", tuple(m2.shape), tuple(z2.shape))

**Next:** `03_evoformer.ipynb` — where `m` and `z` repeatedly exchange information.